In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install nptdms

In [ ]:
# 셀에 그대로 입력 후 실행
!ls "/content/drive/MyDrive/Train Set"

In [ ]:
import os, glob
from pathlib import Path
import numpy as np
import pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from nptdms import TdmsFile

# ─────────────── 0. 환경 설정 ─────────────── #
BASE_DIR    = Path("/content/drive/MyDrive/Train Set")   # ← 문자열 또는 Path 객체
TRAIN_FOLDS = ["Train5","Train4","Train7","Train3","Train2","Train8"]
VAL_FOLD    = "Train1"

FS       = 25_600          # 샘플링 주파수
WIN_SEC  = 3               # 윈도 크기(초)
OVERLAP  = 0.5             # 50 % 겹침
DOWN     = 32              # 다운샘플
WIN      = FS * WIN_SEC
STRIDE   = int(WIN * (1 - OVERLAP))
SEQ_LEN  = WIN // DOWN

TEMP_THRESH   = 180        # 임계치(예시)
TORQUE_THRESH = -17

device = "cuda" if torch.cuda.is_available() else "cpu"

# ─────────────── 1. TDMS 로더 & 컬럼 자동 탐지 ─────────────── #
def load_tdms(fp: Path):
    tdms = TdmsFile.read(fp.as_posix())        # Path → str
    g_vib, g_op = tdms.groups()[0], tdms.groups()[1]
    vib = np.stack([ch.data for ch in g_vib.channels()], axis=0)
    op  = pd.DataFrame({ch.name.strip(): ch.data for ch in g_op.channels()})
    return vib, op

# 예시 파일로 컬럼명 자동 설정
example_dir = BASE_DIR / TRAIN_FOLDS[0]
example_fp  = next(example_dir.glob("*.tdms"))  # 첫 번째 TDMS 파일
_, op_ex = load_tdms(example_fp)

op_ex.columns = op_ex.columns.str.strip()
TEMP_COLS  = [c for c in op_ex.columns if ("Front" in c) or ("Rear" in c)]
TORQUE_COL = next(c for c in op_ex.columns if "Torque" in c)

print("▶ TEMP_COLS :", TEMP_COLS)
print("▶ TORQUE_COL:", TORQUE_COL)

In [ ]:
def compute_true_fail_cycle_for_folder(folder):
    temps_f, temps_r, torques = [], [], []
    for fp in sorted(glob.glob(os.path.join(folder, "*.tdms"))):
        vib, op = load_tdms(fp)
        L = vib.shape[1]
        op = op.rename(columns=str.strip)

        if len(op) == L:
            tf = op[TEMP_COLS[0]].values  # Front
            tr = op[TEMP_COLS[1]].values  # Rear
            tq = op[TORQUE_COL].values
        else:
            tf_val = op[TEMP_COLS[0]].mean()
            tr_val = op[TEMP_COLS[1]].mean()
            tq_val = op[TORQUE_COL].mean()
            tf = np.full(L, tf_val)
            tr = np.full(L, tr_val)
            tq = np.full(L, tq_val)

        temps_f.append(tf)
        temps_r.append(tr)
        torques.append(tq)

    # 시계열 붙이기
    temp_front = np.concatenate(temps_f)
    temp_rear  = np.concatenate(temps_r)
    torque_all = np.concatenate(torques)

    # 디버깅용 분포 출력
    print("▶ Front 온도 분포:", pd.Series(temp_front).describe())
    print("▶ Rear  온도 분포:", pd.Series(temp_rear).describe())
    print("▶ 토크 분포:", pd.Series(torque_all).describe())

    # 10초 단위 cycle 설정
    SAMP = FS * 10
    n_cyc = len(temp_front) // SAMP
    cycles = np.repeat(np.arange(n_cyc), SAMP)

    # 고장 발생 조건
    hit = np.where(
        (temp_front >= TEMP_THRESH) |
        (temp_rear  >= TEMP_THRESH) |
        (torque_all <= TORQUE_THRESH)
    )[0]

    if hit.size == 0:
        return None
    return int(cycles[hit[0]])

# 함수 실행
true_fail_cycle = compute_true_fail_cycle_for_folder(os.path.join(BASE_DIR, VAL_FOLD))
print("▶ True fail cycle (Train8):", true_fail_cycle)

In [ ]:
# ───────────────── 라이브러리 ─────────────────
import numpy as np
import pandas as pd
from pathlib import Path
import glob
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from nptdms import TdmsFile

# ─────────────── 0. 환경 설정 ───────────────
BASE_DIR    = Path("/content/drive/MyDrive/Train Set")     # 반드시 Path!
TRAIN_FOLDS = ["Train5","Train4","Train7","Train3","Train2","Train8"]
VAL_FOLD    = "Train1"

FS       = 25_600
WIN_SEC  = 3
OVERLAP  = 0.5
DOWN     = 32
WIN      = FS * WIN_SEC
STRIDE   = int(WIN * (1 - OVERLAP))
SEQ_LEN  = WIN // DOWN

TEMP_THRESH   = 180
TORQUE_THRESH = -17

device = "cuda" if torch.cuda.is_available() else "cpu"

# ─────────────── 1. TDMS 로더 ───────────────
def load_tdms(fp: Path):
    """
    fp : pathlib.Path 또는 str 모두 허용
    """
    fp = Path(fp)                         # 강제로 Path 변환
    tdms = TdmsFile.read(fp.as_posix())   # TdmsFile 은 str 경로 필요
    g_vib, g_op = tdms.groups()[0], tdms.groups()[1]

    vib = np.stack([ch.data for ch in g_vib.channels()], axis=0)
    op  = pd.DataFrame({
        ch.name.strip(): ch.data for ch in g_op.channels()
    })
    return vib, op

# ─────────────── 2. 컬럼 자동 탐지 ───────────────
example_dir = BASE_DIR / TRAIN_FOLDS[0]
example_fp  = next(example_dir.glob("*.tdms"))   # 첫 번째 TDMS
_, op_ex = load_tdms(example_fp)

op_ex.columns = op_ex.columns.str.strip()
TEMP_COLS  = [c for c in op_ex.columns if ("Front" in c) or ("Rear" in c)]
TORQUE_COL = next(c for c in op_ex.columns if "Torque" in c)

print("▶ TEMP_COLS :", TEMP_COLS)
print("▶ TORQUE_COL:", TORQUE_COL)

# ─────────────── 3. 고장 cycle 계산 함수 ───────────────
def compute_true_fail_cycle_for_folder(folder):
    folder = Path(folder)                           # 문자열 → Path
    temps_f, temps_r, torques = [], [], []

    for fp in sorted(folder.glob("*.tdms")):
        vib, op = load_tdms(fp)
        L = vib.shape[1]
        op = op.rename(columns=str.strip)

        # TDMS 파일마다 운영데이터 길이가 다를 수 있음
        if len(op) == L:                # 1:1 매핑
            tf = op[TEMP_COLS[0]].values     # Front
            tr = op[TEMP_COLS[1]].values     # Rear
            tq = op[TORQUE_COL].values
        else:                           # 운영데이터 평균값으로 채움
            tf = np.full(L, op[TEMP_COLS[0]].mean())
            tr = np.full(L, op[TEMP_COLS[1]].mean())
            tq = np.full(L, op[TORQUE_COL].mean())

        temps_f.append(tf)
        temps_r.append(tr)
        torques.append(tq)

    # 시계열 붙이기
    temp_front = np.concatenate(temps_f)
    temp_rear  = np.concatenate(temps_r)
    torque_all = np.concatenate(torques)

    # 10초 단위 cycle 인덱스
    SAMP   = FS * 10
    n_cyc  = len(temp_front) // SAMP
    cycles = np.repeat(np.arange(n_cyc), SAMP)

    # 고장 조건에 처음 도달한 index
    hit = np.where(
        (temp_front >= TEMP_THRESH) |
        (temp_rear  >= TEMP_THRESH) |
        (torque_all <= TORQUE_THRESH)
    )[0]

    if hit.size == 0:
        return None
    return int(cycles[hit[0]])

# ─────────────── 4. 함수 실행 ───────────────
true_fail_cycle = compute_true_fail_cycle_for_folder(BASE_DIR / VAL_FOLD)
print("▶ True fail cycle (", VAL_FOLD, "):", true_fail_cycle)


In [ ]:
# ─────────────── 3. Streaming Dataset ─────────────── #
class BearingWindowDataset(Dataset):
    def __init__(self, base_dir, folds):
        # TDMS 파일 경로 수집
        self.files = []
        for f in folds:
            self.files += sorted(glob.glob(os.path.join(base_dir, f, "*.tdms")))
        # 파일별 윈도 개수 계산
        self.windows = []
        for fp in self.files:
            vib, _ = load_tdms(fp)
            L = vib.shape[1]
            self.windows.append(max(0, 1 + (L - WIN)//STRIDE))
        # 누적 인덱스 테이블
        self.cum = np.cumsum([0] + self.windows)

    def __len__(self):
        return int(self.cum[-1])

    def __getitem__(self, idx):
        # global idx → 파일‐윈도 인덱스
        file_idx = np.searchsorted(self.cum, idx, side="right") - 1
        win_idx  = idx - self.cum[file_idx]
        fp       = self.files[file_idx]

        vib, _ = load_tdms(fp)
        vib_ds = vib[:, ::DOWN]
        start  = win_idx * (STRIDE//DOWN)
        x      = vib_ds[:, start:start+SEQ_LEN]
        return torch.from_numpy(x.astype(np.float32)), idx

# DataLoader 준비
train_ds     = BearingWindowDataset(BASE_DIR, TRAIN_FOLDS)
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True,  num_workers=2)
val_ds       = BearingWindowDataset(BASE_DIR, [VAL_FOLD])
val_loader   = DataLoader(val_ds,   batch_size=8, shuffle=False, num_workers=2)

print(f"▶ Train windows: {len(train_ds):,}  /  Val windows: {len(val_ds):,}")


In [ ]:
# ─────────────── 4. ConvAE 모델 ─────────────── #
class ConvAE(nn.Module):
    def __init__(self, in_ch, hid=128):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Conv1d(in_ch,  64, 5, stride=2, padding=2), nn.ReLU(),
            nn.Conv1d(64,   128, 5, stride=2, padding=2), nn.ReLU(),
            nn.Conv1d(128, hid,  3, stride=2, padding=1), nn.ReLU(),
        )
        self.dec = nn.Sequential(
            nn.ConvTranspose1d(hid,  128, 4, stride=2, padding=1), nn.ReLU(),
            nn.ConvTranspose1d(128,   64, 4, stride=2, padding=1), nn.ReLU(),
            nn.ConvTranspose1d(64,  in_ch,4, stride=2, padding=1),
        )
    def forward(self, x):
        return self.dec(self.enc(x))

model = ConvAE(in_ch=train_ds[0][0].shape[0]).to(device)
opt   = torch.optim.Adam(model.parameters(), lr=1e-3)
lossf = nn.MSELoss()

In [ ]:
# # ─────────────── 5. 학습 ─────────────── #
# for ep in range(1, 21):
#     model.train()
#     run = 0.0
#     for xb, _ in train_loader:
#         xb = xb.to(device)
#         xh = model(xb)
#         l  = lossf(xh, xb)
#         opt.zero_grad(); l.backward(); opt.step()
#         run += l.item()
#     print(f"Epoch {ep:02d} | recon = {run/len(train_loader):.6f}")

best_val_loss = float('inf')
patience = 5        # 개선 안되는 epoch 허용 수
counter = 0         # 개선되지 않은 epoch 수
early_stop = False

val_losses = []

for ep in range(1, 101):  # max epoch 100으로 넉넉하게 설정
    model.train()
    train_loss = 0.0
    for xb, _ in train_loader:
        xb = xb.to(device)
        xh = model(xb)
        l = lossf(xh, xb)
        opt.zero_grad(); l.backward(); opt.step()
        train_loss += l.item()
    train_loss /= len(train_loader)

    # ── Validation loss 계산 ──
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for xb, _ in val_loader:
            xb = xb.to(device)
            xh = model(xb)
            l = lossf(xh, xb)
            val_loss += l.item()
    val_loss /= len(val_loader)
    val_losses.append(val_loss)

    print(f"Epoch {ep:02d} | Train recon = {train_loss:.6f} | Val recon = {val_loss:.6f}")

    # ── Early Stopping 조건 검사 ──
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        counter = 0
        best_model_wts = model.state_dict()  # 모델 복사
    else:
        counter += 1
        if counter >= patience:
            print(f"▶ Early stopping at epoch {ep} (no improvement for {patience} epochs)")
            early_stop = True
            break

# 조기 종료 후 최적 모델 복원
if early_stop:
    model.load_state_dict(best_model_wts)

In [ ]:
model.eval()
scores, ids = [], []
with torch.no_grad():
    for xb, idx in val_loader:
        xb = xb.to(device)
        xh = model(xb)
        err = ((xb - xh) ** 2).mean(dim=(1, 2)).cpu().numpy()
        scores.append(err); ids.append(idx)
scores = np.concatenate(scores)
ids    = np.concatenate(ids)

hi     = (scores - scores.min()) / (scores.max() - scores.min())
cycles = (ids * STRIDE / FS / WIN_SEC).astype(int)
cy_hi  = (
    pd.DataFrame({"cycle": cycles, "hi": hi})
      .groupby("cycle")
      .mean()
)

total_cycles = cy_hi.index.max()
# true_fail_cycle은 2단계에서 이미 계산해둔 값
RUL_act      = None if true_fail_cycle is None else total_cycles - true_fail_cycle

In [ ]:
th_grid = np.linspace(0.10, 0.90, 17)      # 0.10 ~ 0.90, 간격 0.05
best    = {"th": None, "A_RUL": -np.inf}   # 스코어가 클수록 좋음

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# A_RUL 계산 함수
def rul_score(rul_act, rul_pred):
    if (rul_act is None) or (rul_pred is None):
        return None
    Er = 100.0 * (rul_act - rul_pred) / rul_act
    if Er <= 0:
        return np.exp(-np.log(0.5) * (Er / 20.0))
    else:
        return np.exp(+np.log(0.5) * (Er / 50.0))

# HI Threshold에 따른 A_RUL 평가 및 최적값 탐색
def evaluate_hi_thresholds(cy_hi, RUL_act, total_cycles, th_grid=None):
    if th_grid is None:
        th_grid = np.linspace(0.10, 0.90, 17)

    results = []
    best = {"th": None, "A_RUL": -np.inf, "pred_fail_cycle": None, "RUL_pred": None}

    for th in th_grid:
        pred_cycles = cy_hi[cy_hi.hi > th].index
        pred_fail_cycle = int(pred_cycles.min()) if len(pred_cycles) > 0 else None
        RUL_pred = None if pred_fail_cycle is None else total_cycles - pred_fail_cycle
        A_RUL = rul_score(RUL_act, RUL_pred)

        results.append((th, A_RUL))
        print(f"th={th:.2f}, pred={RUL_pred}, A_RUL={A_RUL}")

        if A_RUL is not None and A_RUL > best["A_RUL"]:
            best.update({
                "th": th,
                "A_RUL": A_RUL,
                "pred_fail_cycle": pred_fail_cycle,
                "RUL_pred": RUL_pred
            })

    return results, best

# A_RUL 시각화 함수
def plot_a_rul_vs_threshold(results, title="A_RUL vs HI_THRESH"):
    thresholds = [r[0] for r in results]
    scores = [r[1] if r[1] is not None else 0 for r in results]

    plt.figure(figsize=(8, 4))
    plt.plot(thresholds, scores, marker='o')
    plt.title(title)
    plt.xlabel("HI Threshold")
    plt.ylabel("A_RUL Score")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

# cy_hi: 사이클별 Health Index DataFrame (예: cy_hi = pd.DataFrame({'cycle': ..., 'hi': ...}))
# RUL_act: 실제 RUL 값
# total_cycles: 전체 사이클 수

results, best = evaluate_hi_thresholds(cy_hi, RUL_act, total_cycles)
plot_a_rul_vs_threshold(results)

sorted_results = sorted(results, key=lambda x: x[1] if x[1] is not None else -np.inf, reverse=True)

print("\n▶ 상위 A_RUL 기준 Top 3 Thresholds")
top_k = 3
for i, (th, score) in enumerate(sorted_results[:top_k]):
    print(f"  Top {i+1}: HI_THRESH = {th:.2f}, A_RUL = {score:.4f}")

print("\n▶ Grid-Search 결과")
print(f"  최적 HI_THRESH   : {best['th']:.2f}")
print(f"  Pred fail cycle  : {best['pred_fail_cycle']}")
print(f"  Pred RUL         : {best['RUL_pred']}")
print(f"  A_RUL(best)      : {best['A_RUL']:.4f}")

In [ ]:
RUL_pred = best["RUL_pred"]

if RUL_act is None or RUL_pred is None:
    Er = None
else:
    Er = 100.0 * (RUL_act - RUL_pred) / RUL_act

# A_RUL 계산
if Er is None:
    A_RUL = None
elif Er <= 0:
    A_RUL = np.exp(-np.log(0.5) * (Er/20.0))
else:
    A_RUL = np.exp(+np.log(0.5) * (Er/50.0))

FinalScore = A_RUL

print(f"Percent Error Er = {Er:.2f}%")
print(f"RUL Score A_RUL = {A_RUL:.4f}")
print(f"Final Score     = {FinalScore:.4f}")